# F-04: Adding R-03 hydrological/thermal lags to the density + partial-pooling models (RFm)

R-03 found **RF_lag** (RF + SWC/TS lagged 1-4 weeks) was the *best* model at **Tower 9** (short/medium gaps) — the tower the pooling work is rescuing. But those lags were never carried into F-01->F-03 (which built on R-02-CO2's lag-free `driver_m` + FCO2). This tests whether re-adding them helps.

Lags (Kim 2020 / R-03, D-23): **SWC and TS each lagged 168/336/504/672 h** (1-4 weeks). 8 columns.

Four variants, all with stocking density (D-29), evaluated on Towers 2/4/9:
- **solo** / **solo+lags** — per-tower model (R-03 RF_lag analog, but on the R-02-CO2 base)
- **partial** / **partial+lags** — partial pooling (pooled + tower dummies, D-30) ± lags

Same training rows across variants (BASE dropna); lag columns mean-imputed. R-02 gap harness.

In [1]:
from pathlib import Path
import datetime
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
HOURLY=Path("../../data/Hourly"); RESULTS=Path("../../results")
PLAUS_LOW,PLAUS_HIGH=-500,3000
N_REPS,MASK_FRAC=5,0.25
SCENARIOS={"vs":1,"s":4,"m":32,"l":288,"m1":"mixed"}
LSU={"cattle":1.0,"sheep":0.1,"lamb":0.05}
AUX=["_hour_sin","_hour_cos","_doy_sin","_doy_cos"]
LAG_HOURS=[168,336,504,672]
CATCHMENT_AREA_HA={1:4.81,2:6.65,3:6.62,4:7.75,5:6.47,6:3.86,7:2.60,8:7.02,9:7.75,10:1.82,11:1.76,12:1.78,13:1.75,14:1.72,15:1.54}
TOWERS=[2,4,9]

## 1  Configs + load

In [2]:
C4="Catchment 4 After  2013/08/13"
def cfg(t,cat,catstr):
    return {"tower":t,"area_ha":CATCHMENT_AREA_HA[cat],
        "target":f"FCH4_1_1_1 [Tower {t}]","ssitc":f"FCH4_SSITC_TEST_1_1_1 [Tower {t}]",
        "sw":f"SWIN_1_1_1 [Tower {t}]","ta":f"TA_0_0_1 [Tower {t}]","vpd":f"VPD_0_0_1 [Tower {t}]",
        "ppfd":f"PPFD_1_1_1 [Tower {t}]","ustar":f"USTAR_0_0_1 [Tower {t}]","ws":f"WS_0_0_1 [Tower {t}]",
        "rn":f"RN_1_1_1 [Tower {t}]","precip":f"Precipitation (mm) [{catstr}]","ts":"TS_1_1_1 [Tower 9]",
        "swc":f"Soil Moisture @ 10cm Depth (%) [{catstr}]","shf":f"SHF_1_1_1 [Tower {t}]","fc":f"FC_1_1_1 [Tower {t}]",
        "livestock":{sp:f"{sp}_{catstr}" for sp in LSU},
        "train_yrs":[2018] if t==2 else list(range(2018,2022)),
        "test_yrs":[2019] if t==2 else list(range(2022,2024))}
TOWER_CONFIGS={2:cfg(2,2,"Catchment 2"),4:cfg(4,4,C4),9:cfg(9,9,"Catchment 9")}
df=pd.read_csv(HOURLY/"consolidated_hourly.csv",low_memory=False)
df["Datetime"]=pd.to_datetime(df["Datetime"],format="mixed"); df=df.set_index("Datetime")
fco2=pd.read_csv(HOURLY/"fco2_gapfilled.csv",low_memory=False)
fco2["Datetime"]=pd.to_datetime(fco2["Datetime"],format="mixed"); fco2=fco2.set_index("Datetime")
for _t in TOWERS:
    c=f"FC_1_1_1 [Tower {_t}]"
    if c in df.columns: df[c]=fco2[f"FC_gapfilled [Tower {_t}]"]
df_raw=df; print("loaded",df_raw.shape)

loaded (70153, 449)


## 2  Functions

In [3]:
def insert_calendar_gaps(d,target,test_yrs,gh,n_reps=N_REPS,seed=0):
    tm=d.index.year.isin(test_yrs); tt=d.index[tm]; va=d.loc[tm,target].notna().values
    n=len(tt); tn=max(1,int(va.sum()*MASK_FRAC)); rb=np.random.default_rng(seed); reps=[]
    for _ in range(n_reps):
        rng=np.random.default_rng(int(rb.integers(0,2**31))); occ=np.zeros(n,bool); m=0
        for sp in rng.permutation(n):
            if m>=tn: break
            g=int(rng.choice([1,4,32,288])) if gh=="mixed" else gh; ep=min(int(sp)+g,n)
            if occ[sp:ep].any(): continue
            occ[sp:ep]=True; m+=int(va[sp:ep].sum())
        reps.append(tt[occ & va])
    return reps

def generic_frame(t):
    c=TOWER_CONFIGS[t]; d=df_raw.copy(); tgt=c["target"]
    d.loc[~d[c["ssitc"]].isin([0,1]),tgt]=np.nan
    d.loc[d[tgt].notna() & ~d[tgt].between(PLAUS_LOW,PLAUS_HIGH,inclusive="both"),tgt]=np.nan
    h,doy=d.index.hour,d.index.dayofyear
    d["_hour_sin"]=np.sin(2*np.pi*h/24); d["_hour_cos"]=np.cos(2*np.pi*h/24)
    d["_doy_sin"]=np.sin(2*np.pi*doy/365); d["_doy_cos"]=np.cos(2*np.pi*doy/365)
    liv=c["livestock"]; area=c["area_ha"]
    lsu=sum(d[liv[s]].fillna(0)*w for s,w in LSU.items())
    g=pd.DataFrame(index=d.index); g["target"]=d[tgt]
    for k in ["sw","ta","vpd","ppfd","ustar","ws","rn","precip","ts","swc","shf","fc"]: g[k]=d[c[k]]
    for a in AUX: g[a]=d[a]
    g["lsu_dens"]=lsu/area
    g["graze"]=(sum(d[liv[s]].fillna(0) for s in LSU)>0).astype(float)
    # R-03 lags: SWC & TS at 1-4 weeks (D-23)
    for lag in LAG_HOURS:
        g[f"swc_lag{lag}"]=g["swc"].shift(lag)
        g[f"ts_lag{lag}"]=g["ts"].shift(lag)
    for tt in TOWERS: g[f"is_t{tt}"]=1.0 if tt==t else 0.0
    g["_year"]=d.index.year
    return g

BASE=["sw","ta","vpd","ppfd","ustar","ws","rn","precip","ts","swc","shf","fc"]+AUX+["lsu_dens","graze"]
LAGC=[f"swc_lag{l}" for l in LAG_HOURS]+[f"ts_lag{l}" for l in LAG_HOURS]
DUM=[f"is_t{t}" for t in TOWERS]

def fit(feat,train_df):
    imp=SimpleImputer(strategy="mean")
    rf=RandomForestRegressor(n_estimators=500,min_samples_leaf=5,n_jobs=-1,random_state=42)
    rf.fit(imp.fit_transform(train_df[feat].values),train_df["target"].values)
    return rf,imp
def eval_on(rf,imp,feat,g,c):
    recs=[]; rg={sc:insert_calendar_gaps(g,"target",c["test_yrs"],gh) for sc,gh in SCENARIOS.items()}
    for sc,reps in rg.items():
        for rep,ts in enumerate(reps):
            if len(ts)<5: continue
            yt=g.loc[ts,"target"].values; yp=rf.predict(imp.transform(g.loc[ts,feat].values))
            recs.append({"scenario":sc,"rep":rep,"R2":r2_score(yt,yp)})
    return pd.DataFrame(recs)

## 3  Train variants

In [4]:
frames={t:generic_frame(t) for t in TOWERS}
# pooled training rows = consistent across variants (BASE dropna)
pool_parts=[]
for t in TOWERS:
    g=frames[t]; yrs=TOWER_CONFIGS[t]["train_yrs"]
    pool_parts.append(g[g["_year"].isin(yrs)][BASE+LAGC+DUM+["target"]].assign(_keep=g[g["_year"].isin(yrs)][BASE+["target"]].notna().all(axis=1)))
pool=pd.concat(pool_parts,ignore_index=True); pool=pool[pool["_keep"]].drop(columns="_keep")
print("pooled train rows:",len(pool))
rf_part,imp_part   = fit(BASE+DUM, pool)
rf_partL,imp_partL = fit(BASE+LAGC+DUM, pool)
solo={}; soloL={}
for t in TOWERS:
    g=frames[t]; yrs=TOWER_CONFIGS[t]["train_yrs"]
    tr=g[g["_year"].isin(yrs)]; tr=tr[tr[BASE+["target"]].notna().all(axis=1)]
    solo[t]=fit(BASE,tr); soloL[t]=fit(BASE+LAGC,tr)
print("trained solo/solo+lag x3, partial, partial+lag")

pooled train rows: 12558


trained solo/solo+lag x3, partial, partial+lag


## 4  Evaluate

In [5]:
rows=[]
for t in TOWERS:
    g=frames[t]; c=TOWER_CONFIGS[t]
    variants={
        "solo":(solo[t][0],solo[t][1],BASE),
        "solo+lags":(soloL[t][0],soloL[t][1],BASE+LAGC),
        "partial":(rf_part,imp_part,BASE+DUM),
        "partial+lags":(rf_partL,imp_partL,BASE+LAGC+DUM),
    }
    for name,(rf,imp,feat) in variants.items():
        m=eval_on(rf,imp,feat,g,c); m["tower"]=f"Tower {t}"; m["variant"]=name; rows.append(m)
res=pd.concat(rows,ignore_index=True); print("eval rows:",len(res))

eval rows: 300


## 5  Results — median R2 (does adding R-03 lags help?)

In [6]:
VAR=["solo","solo+lags","partial","partial+lags"]
for t in TOWERS:
    sub=res[res.tower==f"Tower {t}"]
    tbl=sub.groupby(["variant","scenario"])["R2"].median().unstack("scenario").reindex(index=VAR).reindex(columns=list(SCENARIOS)).round(3)
    print(f"\n=== Tower {t} ==="); print(tbl.to_string())
print("\n--- Overall median R2 (across scenarios) ---")
ov=res.groupby(["tower","variant"])["R2"].median().unstack("variant").reindex(columns=VAR).round(3)
print(ov.to_string())
print("\n--- lag effect (Δ vs no-lag) ---")
for t in TOWERS:
    s=res[res.tower==f"Tower {t}"].groupby("variant")["R2"].median()
    print(f"Tower {t}: solo {s['solo+lags']-s['solo']:+.3f} | partial {s['partial+lags']-s['partial']:+.3f}")


=== Tower 2 ===
scenario         vs      s      m      l     m1
variant                                        
solo         -0.712 -0.437 -0.426 -0.474 -0.141
solo+lags    -0.694 -0.447 -0.415 -0.453 -0.129
partial      -0.245 -0.125 -0.216 -0.201 -0.201
partial+lags -0.074 -0.059 -0.102 -0.069 -0.025

=== Tower 4 ===
scenario         vs      s      m      l     m1
variant                                        
solo          0.248  0.147  0.180  0.007 -0.086
solo+lags     0.241  0.159  0.184  0.019 -0.053
partial       0.238  0.153  0.236  0.028 -0.107
partial+lags  0.244  0.156  0.244  0.052 -0.098

=== Tower 9 ===
scenario         vs      s      m      l     m1
variant                                        
solo         -0.054 -0.046 -0.010 -0.145 -0.008
solo+lags    -0.080 -0.054 -0.019 -0.123 -0.017
partial       0.293  0.220  0.292  0.204  0.251
partial+lags  0.291  0.247  0.295  0.200  0.247

--- Overall median R2 (across scenarios) ---
variant   solo  solo+lags  partial  par

## 6  Append to benchmarks.csv

In [7]:
bench=RESULTS/"benchmarks.csv"; today=datetime.date.today().isoformat()
ex=pd.read_csv(bench); ex=ex[ex["replication"]!="F-04"]
out=[]
for r in res.to_dict("records"):
    out.append({"replication":"F-04","model":f"RFm_{r['variant'].replace('+','_')}","tower":r["tower"],
        "feature_set":"density+SWC/TS_lags" if "lags" in r["variant"] else "density",
        "scenario":r["scenario"],"rep":r["rep"],"split":"pool_T2+4+9" if "partial" in r["variant"] else "solo",
        "R2":round(r["R2"],4),"date":today,"notes":"F04 add R-03 SWC/TS 1-4wk lags to density/partial-pool (D-23/D-30)"})
new=pd.DataFrame(out); comb=pd.concat([ex,new],ignore_index=True); comb.to_csv(bench,index=False)
print(f"Wrote {len(new)} F-04 rows. Total {len(comb)}.")

Wrote 300 F-04 rows. Total 2365.
